In [26]:
import os
import logging
import json

# autoreload
%load_ext autoreload
%autoreload 2

from oracle.config import Config
from oracle.data.ontology_loader import OntologyLoader
from oracle.extract.concept_extractor import ConceptExtractor
from oracle.match.level_selector import LevelSelector
from oracle.match.traversal import TraversalController
from oracle.verify.verifier import Verifier
from oracle.utils.logging import setup_logging


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
OPENAI_API_KEY="insert_your_openai_api_key_here"
OPENAI_BASE_URL="insert_your_openai_base_url_here"  # e.g., "https://api.openai.com/v1"

In [28]:
def classify_module(
    module_title: str,
    module_description: str,
    ontology_path: str = "ontology/msc",
    root_file: str = "msc_entry.json",
    ontology_name: str = "MSC2020",
    llm_model: str = "gpt-4.1-mini",
    llm_temperature: float = 0.0,
    api_key: str = os.getenv("OPENAI_API_KEY"),
    base_url: str = os.getenv("OPENAI_BASE_URL"),
    log_level: str = "INFO",
    langsmith_project: str = "oracle-classification",
    langsmith_api_key: str = None,
    debug_log_dir: str = "logs/",
    depth_limit: int = 10,
):
    # Load config
    config = Config.load_from_args(
        ontology_path=ontology_path,
        llm_model=llm_model,
        llm_temperature=llm_temperature,
        log_level=log_level,
        api_key=api_key,
        base_url=base_url,
        langsmith_project=langsmith_project,
        langsmith_api_key=langsmith_api_key
    )

    # Determine modes
    log_level_str = config.log_level.upper()
    langsmith_mode = log_level_str == "LANGSMITH"
    debug_mode = log_level_str == "DEBUG"
    
    # Setup logging
    if debug_mode:
        log_level_enum = logging.DEBUG
    elif langsmith_mode:
        log_level_enum = logging.INFO  # Use INFO for LangSmith to reduce noise
    else:
        log_level_enum = getattr(logging, log_level_str, logging.INFO)
    
    # Always enable component file logging (files always log at DEBUG level)
    setup_logging(log_level_enum, config=config, log_dir=debug_log_dir)
    
    if langsmith_mode:
        logging.info("LANGSMITH mode enabled - LangSmith tracing will be active")
        logging.info(f"LangSmith project: {config.langsmith_project}")
        logging.info(f"Component logs will be written to: {debug_log_dir} (always DEBUG level)")
    elif debug_mode:
        logging.info("DEBUG mode enabled - File logging and print statements active")
        logging.info(f"Component logs will be written to: {debug_log_dir} (always DEBUG level)")
    else:
        logging.info(f"Component logs will be written to: {debug_log_dir} (always DEBUG level)")
    
    # Initialize components
    llm = config.get_llm()                              # The model to use for classification and verification
    loader = OntologyLoader(ontology_path)              # The ontology loader for loading the ontology
    extractor = ConceptExtractor(llm)                   # Concept extractor for extracting concepts from the module description
    selector = LevelSelector(llm)                       # Level selector for selecting the appropriate level of the ontology at each step
    controller = TraversalController(selector, loader)  # Traversal controller for traversing the ontology in a hierarchical manner
    verifier = Verifier(llm)                            # Verifier for verifying the classification, checking for duplicates and near-duplicates, and ensuring that all core topics, methods, and applications are matched

    # Step 1: Concept Extraction
    print("=" * 30 + "STEP 1: CONCEPT EXTRACTION" + "=" * 30)
    print("Extracting concepts...")
    concept_summary = extractor.extract_concepts(
        title=module_title, 
        text=module_description, 
        langsmith_mode=langsmith_mode,
    )
    print(concept_summary)
    
    # Step 2: Hierarchical Traversal
    print("\n" + "=" * 30 + "STEP 2: HIERARCHICAL TRAVERSAL" + "=" * 30)
    print("Traversing ontology...")
    selected_entries = controller.traverse(
        concept_summary=concept_summary,
        root_file=root_file,
        langsmith_mode=langsmith_mode,
        depth_limit=depth_limit,
    )

    print(selected_entries)
    
    # Step 3: Verification
    print("\n" + "=" * 30 + "STEP 3: VERIFICATION" + "=" * 30)
    print("Verifying selection...")
    final_classification = verifier.verify_selection(
        concept_summary=concept_summary,
        selected_entries=selected_entries,
        ontology_name=ontology_name,
        langsmith_mode=langsmith_mode,
    )    
    return concept_summary, selected_entries, final_classification



In [ ]:
module_name = "Mathematical Tools for Geophysics and Earth Sciences"
module_description = """
Description: Teaches a variety of important and fundamental university-level mathematical tools to tackle mathematical
problems that commonly arise in Earth Sciences such as in planetary sciences, geodynamics, seismic techniques, numerical
modelling, physical and surface processes, tectonics of the ocean and many more... \n\n Learning Outcomes Upon successfully
completing this module, students will be able to:  Transform between co-ordinate systems; Manipulate vectors and matrices
using simple algebra; Solve small systems of linear equations; Solve small eigenvalue problems; Solve simple differential
equations analytically; Differentiate and integrate functions of two independent variables; Compute the gradient of a scalar
field and the div and curl of a vector field; Analyse functions, sequences and series for convergence; \n\n Module Content Syllabus:
Co-ordinate systems and vectors: transformation between co-ordinate systems (Cartesian, polar, cylindrical, spherical); definition of
vector, vector algebra, scalar product, vector product. Multivariable Calculus: interpretation and visualisation of functions of two
independent variables; partial differentiation; chain rule; double integrals. Vector calculus: scalar and vector fields; operators
(gradient, divergence, curl, Laplacian). Matrices, matrix algebra, determinants, linear simultaneous equations, inverse matrices, 
Cramer's rule. Strain matrices and Eigenvalue problem: characteristic polynomials; eigenvalues and eigenvectors; symmetric matrices;
multiple eigenvalues. Infinite Series: Sequences; series; notation; partial sums; convergence; power series. First-order differential
equations: classification of differential equations (DEs), solution by separation of variables, integrating factors, Earth Science examples.
Introduction to modelling: numerical differentiation and integration, approximation, numerical solution of ODEs, forward and backward Euler,
Newton-Raphson iteration.",
"""

ontology_path = "../ontology/msc"
root_file = "msc_entry.json"
ontology_name = "MSC2020"
llm_model = "gpt-oss:20b"
llm_temperature = 0.0   
langsmith_api_key = "insert_your_langsmith_api_key_here"

summ, entries, result = classify_module(
    module_title=module_name,
    module_description=module_description,
    ontology_path=ontology_path,
    root_file=root_file,
    ontology_name=ontology_name,
    llm_model=llm_model,
    llm_temperature=llm_temperature,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    log_level="DEBUG",
    langsmith_api_key=langsmith_api_key,
    depth_limit=2,
)

2025-11-28 22:03:38,293 - root - INFO - Component logging enabled for extractor: logs/extractor.log (always DEBUG level)
2025-11-28 22:03:38,323 - root - INFO - Component logging enabled for traversal: logs/traversal.log (always DEBUG level)
2025-11-28 22:03:38,330 - root - INFO - Component logging enabled for verifier: logs/verifier.log (always DEBUG level)
2025-11-28 22:03:38,331 - root - INFO - Component logging enabled for ontology_loader: logs/ontology_loader.log (always DEBUG level)
2025-11-28 22:03:38,331 - root - INFO - DEBUG mode enabled - File logging and print statements active
2025-11-28 22:03:38,331 - root - INFO - Component logs will be written to: logs/ (always DEBUG level)
2025-11-28 22:03:38,332 - oracle.extract.concept_extractor - INFO - Extracting concepts from text (length: 1969)
2025-11-28 22:03:38,332 - oracle.extract.concept_extractor - INFO -   Input text length: 1969 characters
2025-11-28 22:03:38,333 - root - INFO - Invoking CONCEPT_EXTRACTION_PROMPT
2025-11-2

==============================STEP 1: CONCEPT EXTRACTION==============================
Extracting concepts...


2025-11-28 22:03:42,778 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Server', b'nginx/1.24.0 (Ubuntu)'), (b'Date', b'Fri, 28 Nov 2025 22:03:42 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'1758'), (b'Connection', b'keep-alive'), (b'content-encoding', b'zstd'), (b'vary', b'Accept-Encoding'), (b'x-process-time', b'4')])
2025-11-28 22:03:42,778 - httpx - INFO - HTTP Request: POST https://chat.ese.ic.ac.uk/api/chat/completions "HTTP/1.1 200 OK"
2025-11-28 22:03:42,779 - httpcore.http11 - DEBUG - receive_response_body.started request=<Request [b'POST']>
2025-11-28 22:03:42,779 - httpcore.http11 - DEBUG - receive_response_body.complete
2025-11-28 22:03:42,780 - httpcore.http11 - DEBUG - response_closed.started
2025-11-28 22:03:42,781 - httpcore.http11 - DEBUG - response_closed.complete
2025-11-28 22:03:42,781 - openai._base_client - DEBUG - HTTP Response: POST https://chat.ese.ic.ac.uk/api/chat/completions "20

module_title='Mathematical Tools for Geophysics and Earth Sciences' core_topics=['Coordinate systems and transformations', 'Vector algebra and scalar/vector products', 'Multivariable calculus (functions of two variables, partial differentiation, double integrals)', 'Vector calculus (gradient, divergence, curl, Laplacian)', 'Matrix theory (algebra, determinants, inverse matrices, Cramer’s rule)', 'Eigenvalue problems and strain matrices', 'Infinite series, sequences and convergence', 'First‑order differential equations', 'Numerical methods for differentiation, integration and ODEs'] methods=['Coordinate transformation between Cartesian, polar, cylindrical and spherical systems', 'Vector manipulation using algebraic operations', 'Solving linear systems via matrix algebra and Cramer’s rule', 'Determining eigenvalues and eigenvectors from characteristic polynomials', 'Analytical solution of simple differential equations (separation of variables, integrating factors)', 'Computation of scala

2025-11-28 22:04:01,816 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Server', b'nginx/1.24.0 (Ubuntu)'), (b'Date', b'Fri, 28 Nov 2025 22:04:01 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'4557'), (b'Connection', b'keep-alive'), (b'content-encoding', b'zstd'), (b'vary', b'Accept-Encoding'), (b'x-process-time', b'19')])
2025-11-28 22:04:01,817 - httpx - INFO - HTTP Request: POST https://chat.ese.ic.ac.uk/api/chat/completions "HTTP/1.1 200 OK"
2025-11-28 22:04:01,818 - httpcore.http11 - DEBUG - receive_response_body.started request=<Request [b'POST']>
2025-11-28 22:04:01,820 - httpcore.http11 - DEBUG - receive_response_body.complete
2025-11-28 22:04:01,821 - httpcore.http11 - DEBUG - response_closed.started
2025-11-28 22:04:01,821 - httpcore.http11 - DEBUG - response_closed.complete
2025-11-28 22:04:01,822 - openai._base_client - DEBUG - HTTP Response: POST https://chat.ese.ic.ac.uk/api/chat/completions "2

[LevelSelectionEntry(code='15', label='Linear and multilinear algebra; matrix theory', depth=1, matched_concepts=['Matrix theory (algebra, determinants, inverse matrices, Cramer’s rule)', 'Eigenvalue problems and strain matrices', 'Solving linear systems via matrix algebra and Cramer’s rule', 'Determining eigenvalues and eigenvectors', 'Manipulate vectors and matrices using algebraic techniques', 'Solve small systems of linear equations', 'Solve eigenvalue problems for matrices'], justification='The module covers matrix theory, linear systems, Cramer’s rule, and eigenvalue problems – all central to entry\xa015.', should_descend=False), LevelSelectionEntry(code='26', label='Real functions', depth=1, matched_concepts=['Vector algebra and scalar/vector products', 'Multivariable calculus (functions of two variables, partial differentiation, double integrals)', 'Coordinate transformation between Cartesian, polar, cylindrical and spherical systems', 'Differentiation and integration of functi

2025-11-28 22:05:42,658 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Server', b'nginx/1.24.0 (Ubuntu)'), (b'Date', b'Fri, 28 Nov 2025 22:05:42 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'5659'), (b'Connection', b'keep-alive'), (b'content-encoding', b'zstd'), (b'vary', b'Accept-Encoding'), (b'x-process-time', b'32')])
2025-11-28 22:05:42,659 - httpx - INFO - HTTP Request: POST https://chat.ese.ic.ac.uk/api/chat/completions "HTTP/1.1 200 OK"
2025-11-28 22:05:42,660 - httpcore.http11 - DEBUG - receive_response_body.started request=<Request [b'POST']>
2025-11-28 22:05:42,661 - httpcore.http11 - DEBUG - receive_response_body.complete
2025-11-28 22:05:42,662 - httpcore.http11 - DEBUG - response_closed.started
2025-11-28 22:05:42,662 - httpcore.http11 - DEBUG - response_closed.complete
2025-11-28 22:05:42,663 - openai._base_client - DEBUG - HTTP Response: POST https://chat.ese.ic.ac.uk/api/chat/completions "2

In [34]:
vars(summ)

{'module_title': 'Mathematical Tools for Geophysics and Earth Sciences',
 'core_topics': ['Coordinate systems and transformations',
  'Vector algebra and scalar/vector products',
  'Multivariable calculus (functions of two variables, partial differentiation, double integrals)',
  'Vector calculus (gradient, divergence, curl, Laplacian)',
  'Matrix theory (algebra, determinants, inverse matrices, Cramer’s rule)',
  'Eigenvalue problems and strain matrices',
  'Infinite series, sequences and convergence',
  'First‑order differential equations',
  'Numerical methods for differentiation, integration and ODEs'],
 'methods': ['Coordinate transformation between Cartesian, polar, cylindrical and spherical systems',
  'Vector manipulation using algebraic operations',
  'Solving linear systems via matrix algebra and Cramer’s rule',
  'Determining eigenvalues and eigenvectors from characteristic polynomials',
  'Analytical solution of simple differential equations (separation of variables, integr

In [35]:
entries

[LevelSelectionEntry(code='15', label='Linear and multilinear algebra; matrix theory', depth=1, matched_concepts=['Matrix theory (algebra, determinants, inverse matrices, Cramer’s rule)', 'Eigenvalue problems and strain matrices', 'Solving linear systems via matrix algebra and Cramer’s rule', 'Determining eigenvalues and eigenvectors', 'Manipulate vectors and matrices using algebraic techniques', 'Solve small systems of linear equations', 'Solve eigenvalue problems for matrices'], justification='The module covers matrix theory, linear systems, Cramer’s rule, and eigenvalue problems – all central to entry\xa015.', should_descend=False),
 LevelSelectionEntry(code='26', label='Real functions', depth=1, matched_concepts=['Vector algebra and scalar/vector products', 'Multivariable calculus (functions of two variables, partial differentiation, double integrals)', 'Coordinate transformation between Cartesian, polar, cylindrical and spherical systems', 'Differentiation and integration of funct

In [36]:
result

FinalClassification(module_title='Mathematical Tools for Geophysics and Earth Sciences', ontology_name='MSC2020', selected_codes=[FinalSelectedCode(code='15', label='Linear and multilinear algebra; matrix theory', depth=1, matched_concepts=['Matrix theory (algebra, determinants, inverse matrices, Cramer’s rule)', 'Eigenvalue problems and strain matrices', 'Solving linear systems via matrix algebra and Cramer’s rule', 'Determining eigenvalues and eigenvectors', 'Manipulate vectors and matrices using algebraic techniques', 'Solve small systems of linear equations', 'Solve eigenvalue problems for matrices'], justification='The module covers matrix theory, linear systems, Cramer’s rule, and eigenvalue problems – all central to entry\xa015.'), FinalSelectedCode(code='26', label='Real functions', depth=1, matched_concepts=['Vector algebra and scalar/vector products', 'Multivariable calculus (functions of two variables, partial differentiation, double integrals)', 'Coordinate transformation b

In [37]:
result.module_title = module_name
metadata = {
    "module_description": module_description,
    "ontology_name": ontology_name,
    "llm_model": llm_model,
    "llm_temperature": llm_temperature,
}

# combine metadata and result
output = {**metadata, **result.model_dump()}
output

{'module_description': '\nDescription: Teaches a variety of important and fundamental university-level mathematical tools to tackle mathematical\nproblems that commonly arise in Earth Sciences such as in planetary sciences, geodynamics, seismic techniques, numerical\nmodelling, physical and surface processes, tectonics of the ocean and many more... \n\n Learning Outcomes Upon successfully\ncompleting this module, students will be able to:  Transform between co-ordinate systems; Manipulate vectors and matrices\nusing simple algebra; Solve small systems of linear equations; Solve small eigenvalue problems; Solve simple differential\nequations analytically; Differentiate and integrate functions of two independent variables; Compute the gradient of a scalar\nfield and the div and curl of a vector field; Analyse functions, sequences and series for convergence; \n\n Module Content Syllabus:\nCo-ordinate systems and vectors: transformation between co-ordinate systems (Cartesian, polar, cylind